# Recipe 04 — Async Pipelines
> Run pipeline builds asynchronously for I/O-bound workloads.

| | |
|---|---|
| **Module** | `anchor.pipeline` |
| **Key classes** | `ContextPipeline.abuild` |
| **Difficulty** | Intermediate |

In [ ]:
import asyncio
from anchor.pipeline import ContextPipeline
from anchor.models import QueryBundle, ContextItem, SourceType
from anchor.formatters import GenericTextFormatter
from anchor.memory import SlidingWindowMemory

## 1 — Why async?
When pipeline steps call external services (vector DBs, reranker APIs),
an async build avoids blocking the event loop and enables concurrency.

## 2 — Set up the pipeline (same as sync)
Configuration is identical — only the final `build` call changes.

In [ ]:
pipeline = ContextPipeline(max_tokens=4096)
pipeline.add_system_prompt("You are a helpful assistant.")
pipeline.with_formatter(GenericTextFormatter())

memory = SlidingWindowMemory(max_tokens=1024)
memory.add_turn("user", "What is async programming?")
memory.add_turn("assistant", "Async programming lets you run I/O tasks concurrently.")
pipeline.with_memory(memory)

print("Pipeline configured")

## 3 — Build asynchronously
Use `await pipeline.abuild(query)` inside an async function.
In a Jupyter notebook you can `await` directly at the top level.

In [ ]:
async def build_context():
    query = QueryBundle(query_str="Tell me more")
    result = await pipeline.abuild(query)
    return result

# In a notebook you can simply: result = await pipeline.abuild(query)
# Outside notebooks use asyncio.run():
result = asyncio.run(build_context())

print(f"Items   : {len(result.window.items)}")
print(f"Tokens  : {result.window.used_tokens}")
print(f"Utilization: {result.window.utilization:.1%}")

## 4 — Parallel builds
You can build multiple context windows concurrently with `asyncio.gather`.

In [ ]:
async def parallel_builds():
    queries = [
        QueryBundle(query_str="What is RAG?"),
        QueryBundle(query_str="Explain embeddings"),
        QueryBundle(query_str="Token budgeting tips"),
    ]
    results = await asyncio.gather(
        *(pipeline.abuild(q) for q in queries)
    )
    return results

results = asyncio.run(parallel_builds())

for i, r in enumerate(results):
    print(f"Build {i}: {r.window.used_tokens} tokens, "
          f"{len(r.window.items)} items")

## 5 — Mixing sync and async steps
Anchor automatically wraps synchronous step functions when running
inside `abuild`, so you can mix both freely.

In [ ]:
from anchor.pipeline import PipelineStep

# Sync step — works inside abuild too
def tag_items(items, query, **kwargs):
    for item in items:
        item.content = f"[tagged] {item.content}"
    return items

pipeline.add_step(PipelineStep(name="tagger", fn=tag_items))

result = asyncio.run(build_context())
print(f"Items after async+sync mix: {len(result.window.items)}")

## Key Takeaways
- `await pipeline.abuild(query)` is the async counterpart of `build()`.
- Use `asyncio.gather` to build multiple windows in parallel.
- Sync step functions work inside `abuild` without changes.

**Next:** [Pipeline Diagnostics](05_pipeline_diagnostics.ipynb)